# replace-final-head — faded example 3: Verify the Old Head Parameters Are No Longer in model.parameters() (Faded)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `replace-final-head`. Running the beacon reports progress on the `Transfer: Replace final head` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Transfer: Replace final head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`replace-final-head`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "replace-final-head"
DD_SUBTOPIC = "Transfer: Replace final head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

After replacing `model.fc`, the old head's parameters are completely removed from the model. Assigning a new module to `model.fc` deregisters the old one — its weights are no longer returned by `model.parameters()`. You can confirm this by collecting parameter IDs before and after the swap.

## Faded exercise 3

The swap is done. Your task is to **implement `old_params_still_present(model, old_param_ids)`** — a function that returns `True` if any parameter ID from the old head's `id(p)` set still appears in `model.parameters()`, and `False` otherwise (confirming the old head is gone).

**Fill in:** Return True if any id in old_param_ids matches any id(p) for p in model.parameters(), else False.

In [ ]:
import torch as t
import torch.nn as nn

class ToyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(nn.Linear(8, 16), nn.ReLU())
        self.fc = nn.Linear(16, 1000)
    def forward(self, x):
        return self.fc(self.backbone(x))

def old_params_still_present(model: nn.Module, old_param_ids: set) -> bool:
    raise NotImplementedError()  # TODO: Return True if any id in old_param_ids matches any id(p) for p in model.parameters(), else False.


def _test():
    import torch as t
    t.manual_seed(0)
    model = ToyNet()
    # Collect old head parameter ids before swap
    old_ids = {id(p) for p in model.fc.parameters()}
    # Perform the swap
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 10)
    # Old params should be gone
    result = old_params_still_present(model, old_ids)
    assert result == False, 'Old head parameters are still in model.parameters()!'
    # New head params should be present
    new_ids = {id(p) for p in model.fc.parameters()}
    assert old_params_still_present(model, new_ids), 'Sanity: new params should be present'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class ToyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(nn.Linear(8, 16), nn.ReLU())
        self.fc = nn.Linear(16, 1000)
    def forward(self, x):
        return self.fc(self.backbone(x))

def old_params_still_present(model: nn.Module, old_param_ids: set) -> bool:
    current_ids = {id(p) for p in model.parameters()}
    return bool(old_param_ids & current_ids)
```
</details>